# Automatic Differentiation

Recall from that section 
that derivatives drive all the optimization algorithms
that we will use to train deep networks.
While the calculations are straightforward,
working them out by hand can be tedious and error-prone, 
and these issues only grow
as our models become more complex.

Fortunately all modern deep learning frameworks
take this work off our plates
by offering *automatic differentiation*
(often shortened to *autograd*). 
As we pass data through each successive function,
the framework builds a *computational graph* 
that tracks how each value depends on others.
To calculate derivatives, 
automatic differentiation 
works backwards through this graph
applying the chain rule. 
The computational algorithm for applying the chain rule
in this fashion is called *backpropagation*.

Autograd has a long history:
the earliest references date back
over half a century [@Wengert.1964],
and reverse mode, the variant that powers
modern backpropagation, was developed
by @Linnainmaa.1970 .
that section
recounts this history in full.
Before exploring methods, 
let's first master the autograd package.

In [ ]:
import tensorflow as tf

## Mechanics

We begin with the basic workflow (attach a gradient,
record a computation, run the backward pass), first
on a scalar-valued function, then on vector-valued ones.

### A Simple Function

Let's assume that we are interested
in differentiating the function
$y = 2\mathbf{x}^{\top}\mathbf{x}$
with respect to the column vector $\mathbf{x}$.
To start, we assign `x` an initial value.

In [ ]:
x = tf.range(4, dtype=tf.float32)
x

<!-- d2l:prose id=autograd-md-3 fw=mxnet, pytorch, tensorflow -->

Before we calculate the gradient
of $y$ with respect to $\mathbf{x}$,
we need a place to store it.
In general, we avoid allocating new memory
every time we take a derivative 
because deep learning requires 
successively computing derivatives
with respect to the same parameters
a great many times,
and we might risk running out of memory.
Note that the gradient of a scalar-valued function
with respect to a vector $\mathbf{x}$
is vector-valued with 
the same shape as $\mathbf{x}$.

In [ ]:
x = tf.Variable(x)

We now calculate our function of `x` and assign the result to `y`.

In [ ]:
# Record all computations onto a tape
with tf.GradientTape() as t:
    y = 2 * tf.tensordot(x, x, axes=1)
y

Recording the operations gives the framework a *computational graph*,
shown in the figure. Its nodes are operations and its
edges carry intermediate values.

![The computational graph for $y = 2\mathbf{x}^\top\mathbf{x}$. The forward pass (black) flows from $\mathbf{x}$ to $y$; reverse-mode automatic differentiation walks the same graph backward (blue), multiplying the local derivative at each node via the chain rule to accumulate $\partial y / \partial \mathbf{x} = 4\mathbf{x}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-comp-graph.svg)

The forward pass evaluates the graph from $\mathbf{x}$ to $y$; to obtain
the gradient, automatic differentiation then traverses it in reverse,
multiplying the local derivatives along the way. We unpack computational
graphs and backpropagation in full in that section,
and the underlying mathematics (both modes of automatic differentiation
and their costs) is developed in
that section; for now we
simply use the resulting gradients.

We can now calculate the gradient of `y`
with respect to `x` by calling 
the `gradient` method.

In [ ]:
x_grad = t.gradient(y, x)
x_grad

We already know that the gradient of the function $y = 2\mathbf{x}^{\top}\mathbf{x}$
with respect to $\mathbf{x}$ should be $4\mathbf{x}$.
We can now verify that the automatic gradient computation
and the expected result are identical.

In [ ]:
x_grad == 4 * x

Now let's calculate 
another function of `x`
and take its gradient.
Note that TensorFlow resets the gradient buffer 
whenever we record a new gradient.

In [ ]:
with tf.GradientTape() as t:
    y = tf.reduce_sum(x)
t.gradient(y, x)  # Overwritten by the newly calculated gradient

### Backward for Non-Scalar Variables

When `y` is a vector, 
the most natural representation 
of the derivative of  `y`
with respect to a vector `x` 
is a matrix called the *Jacobian*
that contains the partial derivatives
of each component of `y` 
with respect to each component of `x`.
Likewise, for higher-order `y` and `x`,
the result of differentiation could be an even higher-order tensor.

While Jacobians do show up in some
advanced machine learning techniques,
more commonly we want to sum up 
the gradients of each component of `y`
with respect to the full vector `x`,
yielding a vector of the same shape as `x`.
For example, we often have a vector 
representing the value of our loss function
calculated separately for each example among
a batch of training examples.
Here, we just want to sum up the gradients
computed individually for each example.

By default, TensorFlow returns the gradient of the sum.
In other words, rather than returning 
the Jacobian $\partial_{\mathbf{x}} \mathbf{y}$,
it returns the gradient of the sum
$\partial_{\mathbf{x}} \sum_i y_i$.

In [ ]:
with tf.GradientTape() as t:
    y = x * x
t.gradient(y, x)  # Same as y = tf.reduce_sum(x * x)

## Controlling the Graph

Sometimes the graph the framework records
is not the graph we want to differentiate.
The next two subsections show how to prune it
by detaching individual intermediate results, and how to
switch recording off altogether.

### Detaching Computation

Sometimes, we wish to move some calculations
outside of the recorded computational graph.
For example, say that we use the input 
to create some auxiliary intermediate terms 
for which we do not want to compute a gradient. 
In this case, we need to *detach* 
the respective computational graph
from the final result. 
The following toy example makes this clearer: 
suppose we have `z = x * y` and `y = x * x` 
but we want to focus on the *direct* influence of `x` on `z` 
rather than the influence conveyed via `y`. 
In this case, we can create a new variable `u`
that takes the same value as `y` 
but whose *provenance* (how it was created)
has been wiped out.
Thus `u` has no ancestors in the graph
and gradients do not flow through `u` to `x`.
Now consider `z = x * u`.
Because `u` is treated as a constant equal to $x^2$,
the gradient is $\partial z / \partial x = u = x^2$.
Had we *not* detached, so that `z = x * (x * x)` $= x^3$, we
would instead have obtained $\partial z / \partial x = 3x^2$.

In [ ]:
# Set persistent=True to preserve the compute graph. 
# This lets us run t.gradient more than once
with tf.GradientTape(persistent=True) as t:
    y = x * x
    u = tf.stop_gradient(y)
    z = u * x

x_grad = t.gradient(z, x)
x_grad == u

Note that while this procedure
detaches `y`'s ancestors
from the graph leading to `z`, 
the computational graph leading to `y` 
persists and thus we can calculate
the gradient of `y` with respect to `x`.

In [ ]:
t.gradient(y, x) == 2 * x

### Turning Off Gradient Tracking

Recording operations for a backward pass costs time and memory. When we
only need a value (at prediction time, or while updating parameters by
hand), we can skip the bookkeeping entirely.

TensorFlow only records operations executed inside a `tf.GradientTape`
(a *tape* is the recorded list of executed operations), so
any computation outside a tape is already untracked. To pause recording
*within* a tape, use `tape.stop_recording()`.

In [ ]:
# Outside any GradientTape, nothing is recorded
y = 2 * tf.tensordot(x, x, axes=1)
y

This untracked mode is the default for inference and evaluation
throughout the rest of the book.

## Beyond the Basics

Automatic differentiation is more general
than the fixed formulas we have differentiated so far:
it handles arbitrary control flow, derivatives of derivatives,
and even lets us choose the *direction*
in which the graph is traversed.

### Gradients and Python Control Flow

So far we reviewed cases where the path from input to output 
was well defined via a function such as `z = x * x * x`.
Programming offers us a lot more freedom in how we compute results. 
For instance, we can make them depend on auxiliary variables 
or condition choices on intermediate results. 
One benefit of using automatic differentiation
is that even if building the computational graph of 
a function required passing through a maze of Python control flow
(e.g., conditionals, loops, and arbitrary function calls),
we can still calculate the gradient of the resulting variable.
To illustrate this, consider the following code snippet where 
the number of iterations of the `while` loop
and the evaluation of the `if` statement
both depend on the value of the input `a`.

In [ ]:
def f(a):
    b = a * 2
    while tf.norm(b) < 1000:
        b = b * 2
    if tf.reduce_sum(b) > 0:
        c = b
    else:
        c = 100 * b
    return c

Below, we call this function, passing in a random value, as input.
Since the input is a random variable, 
we do not know what form 
the computational graph will take.
However, whenever we execute `f(a)` 
on a specific input, we realize 
a specific computational graph
and can subsequently run `backward`.

In [ ]:
a = tf.Variable(tf.random.normal(shape=()))
with tf.GradientTape() as t:
    d = f(a)
d_grad = t.gradient(d, a)
d_grad

Even though our function `f` is, for demonstration purposes, a bit contrived,
its dependence on the input is quite simple: 
it is a *linear* function of the scalar `a` 
with piecewise defined scale. 
As such, `f(a) / a` is a constant 
and, moreover, it needs to match 
the gradient of `f(a)` with respect to `a`.

In [ ]:
d_grad == d / a

Dynamic control flow is very common in deep learning. 
For instance, when processing text, the computational graph
depends on the length of the input. 
In these cases, automatic differentiation 
is necessary for statistical modeling 
since it is impossible to compute the gradient *a priori*. 

### Higher-Order Derivatives

Occasionally we need the derivative of a derivative: the curvature of a
function, or the Hessian--vector products (products of the matrix of
second derivatives, the *Hessian*, with a vector) used by some optimizers.
Autograd can differentiate *through* a gradient computation. Take
$f(x) = x^3$, for which $f'(x) = 3x^2$ and $f''(x) = 6x$.

Nest two `GradientTape`s: the outer tape differentiates the gradient
computed under the inner tape.

In [ ]:
x3 = tf.Variable(2.0)
with tf.GradientTape() as outer:
    with tf.GradientTape() as inner:
        y = x3 ** 3
    dy = inner.gradient(y, x3)   # 3x^2 = 12
d2y = outer.gradient(dy, x3)     # 6x  = 12
dy, d2y

### Forward versus Reverse Mode

Automatic differentiation can traverse the computational graph in either
direction. *Reverse mode*, the variant we have used so far and the engine
behind backpropagation, sweeps from the output back to the inputs,
yielding the gradient of a single scalar with respect to *all* inputs in
one pass. *Forward mode* sweeps the other way, propagating derivatives
from one input outward to every output.

The choice is about cost, and a counting argument settles it.
For a function with $n$ inputs and $m$ outputs,
filling the full matrix of derivatives takes
one reverse sweep per *output* ($m$ sweeps)
or one forward sweep per *input* ($n$ sweeps),
each sweep costing about as much as one evaluation of the function.
A training loss is a single scalar ($m = 1$)
depending on millions of parameters ($n$ huge),
so reverse mode delivers the entire gradient
for the price of roughly one extra forward pass.
Forward mode wins in the opposite regime
(few inputs, many outputs), and it is also the tool of choice
for Hessian--vector products and per-input sensitivities,
as in the Julia package ForwardDiff.jl
[@Revels.Lubin.Papamarkou.2016].
The exercises explore this trade-off further, and
that section derives both modes
and their costs in full.

## Discussion

Automatic differentiation frees practitioners
from deriving gradients by hand,
and it makes it practical to train models
for which pen and paper gradient computations 
would be prohibitively time consuming.
While we use autograd to *optimize* models
(in a statistical sense),
the *optimization* of autograd libraries themselves
(in a computational sense)
is a rich subject that matters to framework designers.
Here, tools from compilers and graph manipulation 
are used to compute results 
quickly and with modest memory. 

For now, try to remember these basics:
(i) attach gradients to those variables with respect to which we desire derivatives;
(ii) record the computation of the target value;
(iii) execute the backpropagation function; and
(iv) access the resulting gradient.


## Exercises

1. Why is the second derivative much more expensive to compute than the first derivative?
1. After running the function for backpropagation, immediately run it again and see what happens. Investigate.
1. In the control flow example where we calculate the derivative of `d` with respect to `a`, what would happen if we changed the variable `a` to a random vector or a matrix? At this point, the result of the calculation `f(a)` is no longer a scalar. What happens to the result? How do we analyze this?
1. Let $f(x) = \sin(x)$. Plot the graph of $f$ and of its derivative $f'$. Do not exploit the fact that $f'(x) = \cos(x)$ but rather use automatic differentiation to get the result. 
1. Let $f(x) = ((\log x^2) \cdot \sin x) + x^{-1}$. Write out a dependency graph tracing results from $x$ to $f(x)$. 
1. Use the chain rule to compute the derivative $\frac{df}{dx}$ of the aforementioned function, placing each term on the dependency graph that you constructed previously. 
1. Given the graph and the intermediate derivative results, you have a number of options when computing the gradient. Evaluate the result once sweeping from $x$ to $f$ (forward mode) and once from $f$ tracing back to $x$ (reverse mode). 
1. For the graph of exercise 5, count the operations that forward mode and reverse mode each perform, and the intermediate values each must store. How would the comparison change for a function with many inputs, or with many outputs?

[Discussions](https://d2l.discourse.group/t/200)